# 04. Multi-Stage Temporal Claim Verification & Hallucination Detection

Demonstration of the core verification engine:
1. **Multi-Stage Verification**: Entity existence, relation validation, timestamp distance calculation, and chronological ordering.
2. **Hallucination Detection**: Categorizing claim validity into SUPPORTED, PARTIALLY_SUPPORTED, UNSUPPORTED, and TEMPORALLY_IMPOSSIBLE.
3. **Explanation Generation**: Automated human-readable justifications.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config import settings
from src.knowledge_graph.memory_store import InMemoryKGStore
from src.verification.temporal_verifier import TemporalVerifier
from src.verification.verification_result import TemporalClaim
from src.hallucination.detector import HallucinationDetector
from src.explanation.explainer import ExplanationGenerator

## 1. Initializing Verification Components

In [ ]:
store = InMemoryKGStore()
store.load(settings.PROCESSED_DATA_DIR / "knowledge_graph.json")

verifier = TemporalVerifier(graph_store=store, timestamp_tolerance_days=0)
detector = HallucinationDetector()
explainer = ExplanationGenerator()
print("Verification engine ready.")

## 2. Testing Genuine Fact vs Controlled Hallucinations

In [ ]:
# 1. Genuine Fact from ICEWS14
clean_claim = TemporalClaim(
    subject="South Korea",
    predicate="Make a visit",
    object="Mexico",
    timestamp="2014-03-16"
)

# 2. Timestamp Corruption (Temporal Hallucination)
corrupt_ts_claim = TemporalClaim(
    subject="South Korea",
    predicate="Make a visit",
    object="Mexico",
    timestamp="2014-05-01"
)

# 3. Entity Corruption (Factual Hallucination)
corrupt_ent_claim = TemporalClaim(
    subject="South Korea",
    predicate="Make a visit",
    object="FictionalAtlantis",
    timestamp="2014-03-16"
)

claims_to_test = [clean_claim, corrupt_ts_claim, corrupt_ent_claim]

In [ ]:
for idx, claim in enumerate(claims_to_test, 1):
    vr = verifier.verify(claim)
    report = detector.classify_claim(vr)
    explanation = explainer.explain(vr)
    
    print(f"--- Claim #{idx} ---")
    print(f"Triple: ({claim.subject}, {claim.predicate}, {claim.object}, {claim.timestamp})")
    print(f"Status:       {vr.status.value}")
    print(f"Confidence:   {vr.confidence}")
    print(f"Hallucinated: {report['is_hallucination']}")
    print(f"Explanation:  {explanation}")
    print()